In [ ]:
import os
import random
from dataclasses import dataclass
from typing import Dict, Any, Optional

import numpy as np
import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed as hf_set_seed,
)
import evaluate

In [ ]:
# -----------------------------
# 1) Config
# -----------------------------
@dataclass(frozen=True)
class Config:
    model_name: str = "distilbert-base-uncased"
    dataset_name: str = "imdb"          # 예: "imdb", 또는 load_dataset("csv", ...)로 교체
    text_col: str = "text"
    label_col: str = "label"
    max_length: int = 256

    output_dir: str = "./outputs/seqclf"
    seed: int = 42

    # Training
    num_train_epochs: int = 2
    per_device_train_batch_size: int = 16
    per_device_eval_batch_size: int = 32
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    logging_steps: int = 50
    eval_strategy: str = "steps"
    eval_steps: int = 200
    save_steps: int = 200
    save_total_limit: int = 2

    fp16: bool = True               # Ampere 이상이면 bf16 고려
    bf16: bool = False
    gradient_checkpointing: bool = False

    # Dev/Debug (필요 시)
    train_size: Optional[int] = None  # 예: 1000
    eval_size: Optional[int] = None   # 예: 500


def seed_everything(seed: int) -> None:
    # HF 내부에서도 set_seed를 쓰지만, 실무에서는 아래까지 같이 고정하는 편이 많습니다.
    hf_set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# -----------------------------
# 2) Dataset
# -----------------------------
def build_dataset(cfg: Config) -> DatasetDict:
    ds = load_dataset(cfg.dataset_name)

    # imdb: train/test 구조
    # custom csv라면 아래처럼:
    # ds = load_dataset("csv", data_files={"train": train_path, "validation": val_path})

    if cfg.train_size is not None:
        ds["train"] = ds["train"].shuffle(seed=cfg.seed).select(range(cfg.train_size))
    if cfg.eval_size is not None:
        split = "test" if "test" in ds else "validation"
        ds[split] = ds[split].shuffle(seed=cfg.seed).select(range(cfg.eval_size))

    # Trainer 기본은 "validation"을 평가로 씀 → imdb는 "test"를 eval로 쓰도록 맞춰줌
    if "validation" not in ds:
        if "test" in ds:
            ds = DatasetDict(train=ds["train"], validation=ds["test"])
        else:
            raise ValueError("validation/test split이 없습니다. 데이터셋 구성을 확인하세요.")

    return ds

def tokenize_dataset(ds: DatasetDict, tokenizer: AutoTokenizer, cfg: Config) -> DatasetDict:
    def _tokenize(batch: Dict[str, Any]) -> Dict[str, Any]:
        return tokenizer(
            batch[cfg.text_col],
            truncation=True,
            max_length=cfg.max_length,
        )

    # remove_columns: 원문 컬럼은 학습 입력에 필요 없으면 제거 (메모리/속도에 유리)
    remove_cols = [c for c in ds["train"].column_names if c not in (cfg.label_col,)]
    tokenized = ds.map(
        _tokenize,
        batched=True,
        remove_columns=remove_cols,
        desc="Tokenizing",
    )
    tokenized = tokenized.rename_column(cfg.label_col, "labels")
    tokenized.set_format("torch")
    return tokenized

In [ ]:
# -----------------------------
# 3) Metrics
# -----------------------------
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

In [ ]:
# -----------------------------
# 4) Train / Eval
# -----------------------------
def main(cfg: Config) -> None:
    os.makedirs(cfg.output_dir, exist_ok=True)
    seed_everything(cfg.seed)

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
    ds = build_dataset(cfg)
    ds_tok = tokenize_dataset(ds, tokenizer, cfg)

    num_labels = int(ds["train"].features[cfg.label_col].num_classes) if cfg.label_col in ds["train"].features else 2
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=num_labels,
    )

    if cfg.gradient_checkpointing:
        model.gradient_checkpointing_enable()

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    args = TrainingArguments(
        output_dir=cfg.output_dir,
        seed=cfg.seed,

        # train
        num_train_epochs=cfg.num_train_epochs,
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        per_device_eval_batch_size=cfg.per_device_eval_batch_size,
        learning_rate=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
        warmup_ratio=cfg.warmup_ratio,

        # eval/save/log
        evaluation_strategy=cfg.eval_strategy,
        eval_steps=cfg.eval_steps,
        save_steps=cfg.save_steps,
        save_total_limit=cfg.save_total_limit,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        logging_steps=cfg.logging_steps,
        logging_dir=os.path.join(cfg.output_dir, "logs"),

        # perf
        fp16=cfg.fp16,
        bf16=cfg.bf16,
        gradient_checkpointing=cfg.gradient_checkpointing,
        dataloader_num_workers=2,
        report_to=["none"],  # wandb 쓰면 ["wandb"]로
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()
    print("Eval:", metrics)

    # best model + tokenizer 저장
    trainer.save_model(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)

In [ ]:
# -----------------------------
# 5) Inference (예시)
# -----------------------------
@torch.inference_mode()
def predict(texts: list[str], model_dir: str) -> list[int]:
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.eval()

    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=256)
    outputs = model(**inputs)
    preds = outputs.logits.argmax(dim=-1).cpu().tolist()
    return preds


if __name__ == "__main__":
    cfg = Config(
        model_name="distilbert-base-uncased",
        dataset_name="imdb",
        output_dir="./outputs/imdb_distilbert",
        max_length=256,
        num_train_epochs=2,
        fp16=torch.cuda.is_available(),
        bf16=False,
        train_size=5000,   # 실무 개발용 샘플링 (원하면 None)
        eval_size=1000,
    )
    main(cfg)

    # quick test
    print(predict(["this movie was amazing", "boring and too long"], cfg.output_dir))

In [ ]:
# TODO: implement
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# TODO: implement
def build_datasets(tokenizer, max_length: int = 128): # train_path: str, val_path: str
    """
    return train_dataset, val_dataset
    """
    def tokenize_function(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

    ds = load_dataset("imdb")
    # ds = load_dataset(
    #     "csv",
    #     data_files={"train": train_path, "validation": val_path},
    # )
    
    # ds["train"].select(range(100))
    # ds["train"].shuffle(seed=42).select(range(train_size))
    train_size = 50
    val_size = 20
    train_ds = ds["train"].shuffle(seed=42).select(range(train_size))
    val_ds = ds["test"].shuffle(seed=42).select(range(val_size))
    
    train_tok = train_ds.map(tokenize_function, batched=True)
    val_tok = val_ds.map(tokenize_function, batched=True)

    # label -> labels 로 통일
    train_tok = train_tok.rename_column("label", "labels")
    val_tok   = val_tok.rename_column("label", "labels")

    train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    return train_tok, val_tok

# TODO: implement
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
def compute_metrics(eval_pred):
    """
    return dict with 'accuracy' (and/or f1)
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Accuracy 계산
    # acc = accuracy.compute(predictions=predictions, references=labels)
    
    # F1 Score 계산
    # 이진 분류(IMDB 등)라면 기본값으로 충분하지만, 
    # 다중 분류라면 average="weighted" 또는 "macro"를 설정해야 합니다.
    # f1 = f1.compute(predictions=predictions, references=labels, average="binary")
    
    # return {
    #     "accuracy": acc["accuracy"],
    #     "f1": f1["f1"]
    # }
    
    return accuracy.compute(predictions=preds, references=labels)

# TODO: implement
def train(model_name: str ="distilbert-base-uncased", 
          output_dir: str ="./results", 
          num_labels = 2):
    """
    1) load tokenizer/model
    2) build datasets
    3) train
    4) evaluate and save best model
    """
    set_seed(42)
    device = torch.device("cpu")

    # 1)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, 
                                                               num_labels=num_labels) 
    model.to(device)

    # 2)
    train_dataset, val_dataset = build_datasets(tokenizer = tokenizer,
                                                max_length = 128) # train_csv, val_csv,

    # 3)
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",  # steps, epoch
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        num_train_epochs=1,
        save_strategy="no",
        logging_strategy="steps",
        logging_steps=10,
        # report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        compute_metrics=compute_metrics
    )
    trainer.train()

    # 4)
    metrics = trainer.evaluate()
    # print(metrics)

    # trainer.save_model(output_dir)

    return metrics    

# if __name__ == "__main__":
#     train()

In [4]:
train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

/home/jaehyun_park/miniforge3/envs/rl312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.692666,0.710921,0.450000


/home/jaehyun_park/miniforge3/envs/rl312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.7109211087226868, 'eval_accuracy': 0.45, 'eval_runtime': 1.4882, 'eval_samples_per_second': 13.439, 'eval_steps_per_second': 2.016, 'epoch': 1.0}


{'eval_loss': 0.7109211087226868,
 'eval_accuracy': 0.45,
 'eval_runtime': 1.4882,
 'eval_samples_per_second': 13.439,
 'eval_steps_per_second': 2.016,
 'epoch': 1.0}